# Cheat Sheet — Sportsbook Spread Prediction (Extended Project)

Quick-reference syntax for every technique used in this project. Snippets use small dummy data so every cell runs standalone — copy the *pattern* into your working notebook.

Sections: Setup · Inspection · String/Record Cleaning · Feature Engineering · EDA & Leakage Checks · Encoding · Models · Tuning & CV · Evaluation · Feature Importance · Betting Backtest · Persistence.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## 1. Loading & Inspection

In [ ]:
df = pd.read_csv('sports_betting_odds.csv')
df.head()
df.shape
df.info()
df.describe().T
df.describe(include='object').T


## 2. Data Quality Audit

In [ ]:
audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(),
    'n_missing': df.isnull().sum(),
})
audit

df.duplicated().sum()
df.drop_duplicates(inplace=True)

# Normalize inconsistent category casing
df['sportsbook'] = df['sportsbook'].str.title().replace({'Betmgm': 'BetMGM'})
df['sportsbook'].unique()

for c in df.select_dtypes(include='object').columns:
    print(c, '->', df[c].unique()[:8])


## 3. Parsing 'W-L' Records & Unit-Bearing Text

In [ ]:
# "W-L" string -> win percentage
def record_to_winpct(s):
    w, l = s.split('-')
    w, l = int(w), int(l)
    return w / (w + l) if (w + l) > 0 else 0.5

pd.Series(['42-18', '10-2']).apply(record_to_winpct)

# Strip a unit suffix, cast to float
pd.Series(['108.4 pts', '95.0 pts']).str.replace(' pts', '').astype(float)

# Strip a unit prefix/suffix combo ("$2.4M")
pd.Series(['$2.4M', '$0.5M']).str.replace('$', '', regex=False).str.replace('M', '').astype(float)

# Strip thousands-comma + unit suffix ("2,086 miles")
pd.Series(['2,086 miles', '434 miles']).str.replace(',', '').str.replace(' miles', '').astype(float)

# Percent strings
pd.Series(['62%', '45%']).str.replace('%', '').astype(int)


## 4. Disguised-Missing Sentinels + Missing-Flag Pattern

In [ ]:
# "Not Reported" mixed into an otherwise-numeric column
raw = pd.Series(['2 players', 'Not Reported', '0 players'])
cleaned = raw.str.replace(' players', '').replace('Not Reported', '-1').astype(int)

was_missing = (cleaned == -1).astype(int)          # flag column: preserves the "missingness" signal
median_known = cleaned[cleaned != -1].median()
imputed = cleaned.replace(-1, median_known)         # numeric column: safe to feed a model

print(was_missing.tolist(), imputed.tolist())

# Same pattern for a differently-spelled sentinel ("Unknown")
raw2 = pd.Series(['946 miles', 'Unknown'])
cleaned2 = raw2.str.replace(' miles', '').replace('Unknown', np.nan).astype(float)
unknown_flag = cleaned2.isnull().astype(int)
imputed2 = cleaned2.fillna(cleaned2.median())


## 5. Domain-Derived Differential Features

In [ ]:
# Per-team stats -> matchup differentials (this is usually where the real signal lives)
df['net_rating_home'] = df_ppg_home = None  # placeholder pattern illustration only

# Real pattern:
# df['net_rating_home'] = df['home_ppg'] - df['home_papg']
# df['net_rating_away'] = df['away_ppg'] - df['away_papg']
# df['net_rating_diff'] = df['net_rating_home'] - df['net_rating_away']
# df['win_pct_diff']    = df['home_win_pct'] - df['away_win_pct']
# df['rest_advantage']  = df['home_rest_days'] - df['away_rest_days']
# df['injury_advantage']= df['away_injuries'] - df['home_injuries']
# df['sharp_public_divergence'] = df['sharp_bet_pct_home'] - df['public_bet_pct_home']


## 6. EDA & the Betting-Data Leakage Check

In [ ]:
# Correlation of every numeric column with the target
numeric_df = df.select_dtypes(include='number')
numeric_df.corr()['closing_spread_home'].sort_values()

# Explicit check: is a "different snapshot of the same market" hiding in your features?
df['closing_spread_home'].corr(df['opening_spread_home'])       # expect very high
df['closing_spread_home'].corr(df['closing_moneyline_home'])    # expect very high

# Rule of thumb for betting data specifically:
# if a column is just another BOOKMAKER-SET number for the same game/market close to
# the one you're predicting, it's a leakage risk relative to an "independent model" goal
# -- even if it's technically observed "before" your target in some looser sense.

# VIF (multicollinearity) on the features you actually plan to keep
from statsmodels.stats.outliers_influence import variance_inflation_factor
X_vif = numeric_df.drop(columns=['closing_spread_home']).dropna()
vif = pd.DataFrame({
    'feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)


## 7. Leakage-Safe Target Encoding for Team Identity

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
low_card = [c for c in cat_cols if df[c].nunique() < 5]     # e.g. sportsbook
high_card = [c for c in cat_cols if df[c].nunique() >= 5]   # e.g. home_team, away_team

df_enc = pd.get_dummies(df, columns=low_card, drop_first=True)
bool_cols = df_enc.select_dtypes(include='bool').columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)

from sklearn.model_selection import train_test_split
X = df_enc.drop(columns=['closing_spread_home'])   # + drop leakage-risk columns here too
y = df_enc['closing_spread_home']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

global_mean = y_train.mean()
for col in high_card:
    means = y_train.groupby(X_train[col]).mean()
    X_train[col] = X_train[col].map(means)
    X_test[col] = X_test[col].map(means).fillna(global_mean)


## 8. Models

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

y_pred_baseline = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

lin = LinearRegression().fit(X_train, y_train)
ridge = Ridge(alpha=1.0, random_state=RANDOM_STATE).fit(X_train, y_train)
rf = RandomForestRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
gb = GradientBoostingRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)


## 9. Cross-Validation & Tuning

In [ ]:
from sklearn.model_selection import cross_val_score, RandomizedSearchCV

scores = -cross_val_score(lin, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(scores.mean(), scores.std())

param_dist = {'n_estimators': [100, 200, 400], 'max_depth': [None, 5, 10, 20],
              'min_samples_split': [2, 5, 10]}
search = RandomizedSearchCV(RandomForestRegressor(random_state=RANDOM_STATE),
                             param_distributions=param_dist, n_iter=20, cv=5,
                             scoring='neg_mean_absolute_error', random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_train, y_train)
best_model = search.best_estimator_


## 10. Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)
# CAUTION: MAPE is unstable when y_true is near 0, which happens often for near-pick'em spreads
mape = np.mean(np.abs((y_test - y_pred) / y_test.replace(0, np.nan))) * 100

plt.scatter(y_test, y_pred, alpha=0.5)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--')
plt.xlabel('Actual spread'); plt.ylabel('Predicted spread')
plt.show()


## 11. Feature Importance

In [ ]:
importances = best_model.feature_importances_
top_idx = np.argsort(importances)[-10:][::-1]
sns.barplot(x=importances[top_idx], y=X_train.columns[top_idx])
plt.show()

from sklearn.inspection import permutation_importance
result = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_idx = result.importances_mean.argsort()[-10:][::-1]
sns.barplot(x=result.importances_mean[perm_idx], y=X_test.columns[perm_idx])
plt.show()


## 12. Betting Backtest Pattern

In [ ]:
# Reattach identifying/outcome columns (held out of X the whole time) to the test predictions
test_df = df.loc[y_test.index].copy()
test_df['model_spread'] = y_pred
test_df['edge'] = test_df['model_spread'] - test_df['opening_spread_home']

threshold = 2.0
bets = test_df[test_df['edge'].abs() >= threshold].copy()
bets['bet_side'] = np.where(bets['edge'] < 0, 'home', 'away')

def ats_won(row):
    home_covers = row['home_margin'] > -row['closing_spread_home']
    return home_covers if row['bet_side'] == 'home' else (not home_covers)

bets['win'] = bets.apply(ats_won, axis=1)
win_rate = bets['win'].mean()
roi = (bets['win'] * 0.909 - (~bets['win']) * 1.0).mean()   # standard -110 odds
print(f"{len(bets)} bets | win rate {win_rate:.1%} | simulated ROI {roi:+.1%}")

# Compare to break-even win rate at -110: 110 / (110 + 100) = 52.4%
BREAKEVEN_AT_MINUS_110 = 110 / 210


## 13. Persistence & Inference

In [ ]:
import joblib

joblib.dump(best_model, 'spread_model.pkl')
joblib.dump({'target_encoding_maps': {}, 'model_columns': list(X_train.columns)}, 'spread_encoders.pkl')

def predict_fair_spread(raw_game_dict, model, encoders):
    # 1. put raw_game_dict into a one-row DataFrame
    # 2. re-apply the SAME cleaning / feature engineering / encoding used in training
    # 3. reindex to training column order, filling missing dummy columns with 0
    # 4. return model.predict(row)[0]
    pass


## Quick lookup: leakage checklist specific to betting data

| Column type | Safe to use as a feature? | Why |
|---|---|---|
| Team/game performance stats (ppg, records, rest, injuries) | Yes | Known before the market sets any line |
| Market's **opening** line for the SAME market you're predicting the close of | Usually no, if your goal is an independent estimate | Encodes the market's own efficient-market synthesis of everything else |
| Market's **closing** line in a different format (moneyline vs. spread) | No — this is direct leakage | It's mathematically ~equivalent to the target itself |
| Public/sharp betting percentages | Yes | Describes money flow, not the resulting line |
| Final score / game outcome | Only for backtesting, never as a training feature | Not known until after the event the model predicts |
